In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# InceptionV3

# YOLO 

In [42]:
from ultralytics import YOLO
import cv2
import torch

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\soham\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [43]:
model = YOLO("yolov8s.pt")  # Pretrained YOLOv8 model

100%|█████████████████████████████████████████████████████████████████████████████| 21.5M/21.5M [00:02<00:00, 11.1MB/s]


In [3]:
for layer in base_model.layers:
    layer.trainable = False

In [13]:
# Add Custom Fully Connected Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)  # Prevent Overfitting
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

In [28]:
predictions = Dense(1, activation='softmax')(x) 

In [29]:
inception_model = Model(inputs=base_model.input, outputs=predictions)

In [30]:
inception_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [23]:
inception_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 111, 111, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 111, 111, 32)      │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 111, 111, 32)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 109, 109, 32)      │           9,216 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 109, 109, 32)      │              96 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 109, 109, 32)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 109, 109, 64)      │          18,432 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 109, 109, 64)      │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 109, 109, 64)      │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 54, 54, 64)        │               0 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 54, 54, 80)        │             240 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 54, 54, 80)        │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 52, 52, 192)       │         138,240 │ activation_3[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 22,983,714 (87.68 MB)

 Trainable params: 1,180,930 (4.50 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [33]:
train_datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=30, 
    width_shift_range=0.2, 
    height_shift_range=0.2,
    zoom_range=0.2,    
    horizontal_flip=True,  
    validation_split=0.2   
)


train_generator = train_datagen.flow_from_directory(
    directory="OilSpillDetectionAndMonitoring/data/Images/train", 
    target_size=(224, 224), 
    batch_size=32,
    class_mode='categorical',  
    subset='training'
)


val_generator = train_datagen.flow_from_directory(
    directory="OilSpillDetectionAndMonitoring/data/Images/valid",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 624 images belonging to 2 classes.
Found 44 images belonging to 2 classes.


In [36]:
print(f"Train generator: {train_generator}")
print(f"Validation generator: {val_generator}")

Train generator: <keras.src.legacy.preprocessing.image.DirectoryIterator object at 0x00000278924A2F30>
Validation generator: <keras.src.legacy.preprocessing.image.DirectoryIterator object at 0x000002789363BEF0>


In [35]:
# Train the Model
model.fit(train_generator, validation_data=val_generator, epochs=10)

Epoch 1/10


C:\Users\soham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


AttributeError: 'NoneType' object has no attribute 'items'

In [ ]:
model.fit(train_generator, validation_data=val_generator, epochs=5)

In [67]:
train: "D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/train/images"
val: "D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/valid/images"
test: "D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/test/images"

nc: 2
names: ['Look-alike', 'Oil Spill']

In [68]:
import os

dataset_path = "D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images"

for split in ["train", "valid", "test"]:
    img_path = os.path.join(dataset_path, split, "images")
    lbl_path = os.path.join(dataset_path, split, "labels")

    print(f"Checking {split} set:")
    print(f" - Images found: {len(os.listdir(img_path)) if os.path.exists(img_path) else 'Not Found'}")
    print(f" - Labels found: {len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 'Not Found'}\n")


Checking train set:
 - Images found: 779
 - Labels found: 779

Checking valid set:
 - Images found: 221
 - Labels found: 221

Checking test set:
 - Images found: 111
 - Labels found: 111



In [72]:
model.train(data="D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/data.yaml", 
            epochs=10, batch=16, imgsz=640, device="cpu")

Ultralytics 8.3.88  Python-3.12.9 torch-2.4.1+cpu CPU (Intel Core(TM) i5-7300HQ 2.50GHz)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/data.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=None, name=train9, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save

train: Scanning D:\Documents\MIT\Coding\OilSpillDetectionAndMonitoring\data\Images\train\labels.cache... 778 images, 40
val: Scanning D:\Documents\MIT\Coding\OilSpillDetectionAndMonitoring\data\Images\valid\labels.cache... 221 images, 12 b


Plotting labels to runs\detect\train9\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train9
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/10         0G      1.802      3.581      1.611         22        640: 100%|██████████| 49/49 [09:35<00:00, 11.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [01:01<0


                   all        221       1018     0.0042      0.521     0.0762     0.0358

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        221       1018      0.364      0.223      0.209      0.112



10 epochs completed in 1.328 hours.
Optimizer stripped from runs\detect\train9\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train9\weights\best.pt, 6.2MB

Validating runs\detect\train9\weights\best.pt...
Ultralytics 8.3.88  Python-3.12.9 torch-2.4.1+cpu CPU (Intel Core(TM) i5-7300HQ 2.50GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:21<0


                   all        221       1018      0.363      0.223      0.209      0.112
            Look-alike        102        102      0.362      0.186      0.224      0.125
             Oil Spill        172        916      0.365      0.259      0.195     0.0983
Speed: 2.0ms preprocess, 75.2ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs\detect\train9


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000278A8A7A300>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [73]:
metrics = model.val()
print(metrics)

Ultralytics 8.3.88  Python-3.12.9 torch-2.4.1+cpu CPU (Intel Core(TM) i5-7300HQ 2.50GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning D:\Documents\MIT\Coding\OilSpillDetectionAndMonitoring\data\Images\valid\labels.cache... 221 images, 12 b
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:16


                   all        221       1018      0.363      0.223      0.209      0.112
            Look-alike        102        102      0.362      0.186      0.224      0.125
             Oil Spill        172        916      0.365      0.259      0.195     0.0983
Speed: 1.2ms preprocess, 59.3ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs\detect\train92
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000278A863A570>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.01901

In [74]:
metrics = model.val(data="D:/Documents/MIT/Coding/OilSpillDetectionAndMonitoring/data/Images/data.yaml", 
                    split="test")  

print(f"Test mAP50: {metrics.box.map50:.2%}")  # mAP@50
print(f"Test mAP50-95: {metrics.box.map:.2%}")  # mAP@50-95


Ultralytics 8.3.88  Python-3.12.9 torch-2.4.1+cpu CPU (Intel Core(TM) i5-7300HQ 2.50GHz)


val: Scanning D:\Documents\MIT\Coding\OilSpillDetectionAndMonitoring\data\Images\test\labels... 111 images, 2 backgroun


val: New cache created: D:\Documents\MIT\Coding\OilSpillDetectionAndMonitoring\data\Images\test\labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:10<0


                   all        111        461      0.279       0.28      0.202     0.0979
            Look-alike         53         53      0.296      0.245      0.189     0.0868
             Oil Spill         92        408      0.261      0.314      0.215      0.109
Speed: 1.3ms preprocess, 62.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs\detect\train93
Test mAP50: 20.22%
Test mAP50-95: 9.79%


Better

In [ ]:
YOLO_model2 = model.train(data="your_data.yaml", epochs=50, batch=16, imgsz=640, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, flipud=0.5, fliplr=0.5)